# Exercise D — Same Query, Chroma vs Neo4j
Ingest identical chunks+embeddings into two stores and compare retrieval. They should mostly agree — the point is that the store is swappable; quality comes from embeddings + chunking, not the database brand.

**Offline scaffold:** we simulate 'two stores' as two functions over the same vectors. The real exercise wires one to Chroma and one to Neo4j.

In [ ]:
# Offline mock so this scaffold runs with NO API key / NO network.
# For real practice, replace `embed()` with your real embedder (InHouseEmbeddings,
# SentenceTransformer, etc.) and `llm()` with a real model call.
import numpy as np, re
_STOP=set("the a an to of and or is are be for in on at by with from as that this it its".split())
def _tok(t): return [w for w in re.findall(r"[a-z0-9]+",t.lower()) if w not in _STOP and len(w)>2]
def embed(texts):
    if isinstance(texts,str): texts=[texts]
    out=[]
    for t in texts:
        v=np.zeros(256)
        for w in _tok(t): v[abs(hash(w))%256]+=1
        n=np.linalg.norm(v); out.append(v/n if n else v)
    return np.array(out)
def cos(a,b): return float(a@b)

# A small corpus standing in for chunks of ERP-2008-chapter4.pdf (health-care economics).
CORPUS = [
 ("Demand for health care is derived from the value of improved health, not the procedures themselves.","demand"),
 ("Health can be defined by longevity (length of life) and quality of life.","demand"),
 ("National health spending reached over 7000 dollars per capita and about 16 percent of GDP.","spending"),
 ("Medical technology accounts for about half of long-term health spending growth.","spending"),
 ("Medicare, enacted in 1965, covers people aged 65 and older; Part D is the drug benefit.","medicare"),
 ("Medicaid, established in 1965, is a program for low-income individuals, administered by states.","medicaid"),
 ("Moral hazard is the tendency to overuse care when insurance covers most of the cost.","moral_hazard"),
 ("Adverse selection is when insurance is most attractive to those most likely to need it.","insurance"),
 ("Health Savings Accounts use pre-tax dollars with high-deductible plans to reduce routine-care reliance.","hsa"),
 ("The proposed standard deduction for health insurance would be a flat 15000 dollars per family.","tax"),
]
texts=[c[0] for c in CORPUS]; sections=[c[1] for c in CORPUS]
print("Mock corpus ready:", len(texts), "chunks.")

In [ ]:
vectors = embed(texts)

# "Store 1" = brute-force cosine (stands in for Chroma)
def store_chroma(qv, k=3):
    order = sorted(range(len(texts)), key=lambda i: -cos(qv, vectors[i]))[:k]
    return order

# "Store 2" = same math, different code path (stands in for Neo4j's Cypher cosine)
def store_neo4j(qv, k=3):
    sims = [(i, sum(vectors[i][j]*qv[j] for j in range(len(qv)))) for i in range(len(texts))]
    order = [i for i,_ in sorted(sims, key=lambda x:-x[1])[:k]]
    return order

qv = embed("how is health defined?")[0]
c = store_chroma(qv); n = store_neo4j(qv)
print("Chroma top-3 ids:", c)
print("Neo4j  top-3 ids:", n)
print("Agree on:", set(c)&set(n))
for i in c: print(f"  chroma [{i}] {texts[i][:60]}")

### Observe & decide
- Both stores should return the same top chunks (same vectors, same cosine). Where a real Chroma vs Neo4j comparison differs, it's index type or tie-breaking, not 'better retrieval'.
**Your turn:** ingest the real chunks into both stores (exercises 1 and 3) and compare five queries. Builds systems intuition: the vector store is an implementation detail.